# Milestone 1: Data Hygiene, Leakage Prevention & Perceptual Deduplication

This interactive tutorial demonstrates the three critical data hygiene foundations for **MRITumorDetection-CNN**:
1. **Patient-Level Data Leakage**: Why 2D slice-level random splitting fails in clinical imaging, and how `StratifiedGroupKFold` grouped by Patient ID (`PID`) guarantees zero patient leakage.
2. **Cross-Dataset Deduplication**: Why cryptographic hashes (MD5) fail on re-encoded/resized images, and how 64-bit DCT Perceptual Hashing (`pHash`) with a Burkhard-Keller Metric Tree (`BKTree`) identifies cross-dataset duplicates.
3. **Cranial Contour Isolation & Intensity Normalization**: How Otsu thresholding removes variable scanner borders and percentile clipping $[0.5, 99.5]\%$ neutralizes Rician noise spikes.


## 1. Imports and Algorithmic Implementations
We provide self-contained implementations of `compute_phash`, `BKTree`, `crop_brain_contour`, and `normalize_intensity`.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fftpack import dct
import cv2
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

print("Libraries loaded successfully.")


In [ ]:
def compute_phash(image: np.ndarray, hash_size: int = 8, highfreq_factor: int = 4) -> int:
    """
    Computes a 64-bit Discrete Cosine Transform (DCT) perceptual hash.
    1. Downsamples image to 32x32.
    2. Computes 2D DCT-II.
    3. Extracts top-left 8x8 low-frequency submatrix.
    4. Computes median excluding DC term (0,0).
    5. Thresholds into a 64-bit integer.
    """
    img_size = hash_size * highfreq_factor
    if image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.shape[2] == 3 else image[:, :, 0]
    else:
        gray = image.astype(np.float32)
        
    resized = cv2.resize(gray, (img_size, img_size), interpolation=cv2.INTER_AREA)
    dct_2d = dct(dct(resized.T, norm='ortho').T, norm='ortho')
    dct_low = dct_2d[:hash_size, :hash_size]
    
    # Median excluding DC component at (0, 0)
    med = np.median(dct_low.flatten()[1:])
    diff = dct_low > med
    
    hash_val = 0
    for bit in diff.flatten():
        hash_val = (hash_val << 1) | int(bit)
    return hash_val

def hamming_distance(h1: int, h2: int) -> int:
    """Computes bitwise Hamming distance between two 64-bit hashes."""
    return bin(h1 ^ h2).count('1')

class BKTreeNode:
    def __init__(self, hash_val: int, item_id: str):
        self.hash_val = hash_val
        self.item_id = item_id
        self.children = {}  # distance -> BKTreeNode

class BKTree:
    """Burkhard-Keller Metric Tree for fast near-neighbor search under triangle inequality."""
    def __init__(self):
        self.root = None
        self.size = 0

    def insert(self, hash_val: int, item_id: str):
        if self.root is None:
            self.root = BKTreeNode(hash_val, item_id)
            self.size += 1
            return
        
        curr = self.root
        while True:
            dist = hamming_distance(hash_val, curr.hash_val)
            if dist == 0 and curr.item_id == item_id:
                return
            if dist not in curr.children:
                curr.children[dist] = BKTreeNode(hash_val, item_id)
                self.size += 1
                break
            curr = curr.children[dist]

    def search(self, query_hash: int, max_distance: int = 6):
        if self.root is None:
            return []
        candidates = []
        stack = [self.root]
        while stack:
            node = stack.pop()
            d = hamming_distance(query_hash, node.hash_val)
            if d <= max_distance:
                candidates.append((node.item_id, d))
            low = d - max_distance
            high = d + max_distance
            for dist_key, child in node.children.items():
                if low <= dist_key <= high:
                    stack.append(child)
        return candidates


In [ ]:
def crop_brain_contour(image: np.ndarray, margin_ratio: float = 0.05) -> tuple[np.ndarray, tuple]:
    """
    Isolates cranial contour via Otsu thresholding and largest external contour.
    Returns (cropped_image, (x, y, w, h)).
    """
    if image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.shape[2] == 3 else image[:, :, 0]
    else:
        gray = image.copy()
        
    if gray.dtype != np.uint8:
        norm = (gray - gray.min()) / (gray.max() - gray.min() + 1e-8) * 255.0
        gray_u8 = norm.astype(np.uint8)
    else:
        gray_u8 = gray

    blurred = cv2.GaussianBlur(gray_u8, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image, (0, 0, image.shape[1], image.shape[0])

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)

    H, W = image.shape[:2]
    mx = int(w * margin_ratio)
    my = int(h * margin_ratio)
    x0 = max(0, x - mx)
    y0 = max(0, y - my)
    x1 = min(W, x + w + mx)
    y1 = min(H, y + h + my)

    cropped = image[y0:y1, x0:x1]
    return cropped, (x0, y0, x1 - x0, y1 - y0)

def normalize_intensity(image: np.ndarray, lower_percentile: float = 0.5, upper_percentile: float = 99.5) -> np.ndarray:
    """
    Robust intensity normalization via percentile clipping.
    """
    img = image.astype(np.float32)
    non_zeros = img[img > 0]
    if len(non_zeros) == 0:
        return img
    
    p_low = np.percentile(non_zeros, lower_percentile)
    p_high = np.percentile(non_zeros, upper_percentile)
    
    if p_high <= p_low:
        return np.zeros_like(img)
    
    clipped = np.clip(img, p_low, p_high)
    normalized = (clipped - p_low) / (p_high - p_low + 1e-8)
    return normalized


## 2. Synthetic Multi-Slice Patient MRI Simulation
We simulate multi-slice MRI data for 20 patients (each having 4 contiguous 2D axial slices) with patient-specific cranium geometries, ventricles, and tumor pathologies.

In [ ]:
def generate_synthetic_slice(patient_id: str, slice_idx: int, class_id: int, img_size: int = 256) -> np.ndarray:
    """Simulates a patient-specific 2D axial MRI slice."""
    seed = abs(hash(patient_id)) % (2**31) + slice_idx * 17
    rng = np.random.RandomState(seed)
    
    canvas = np.zeros((img_size, img_size), dtype=np.float32)
    center = (img_size // 2, img_size // 2)
    
    patient_base_seed = abs(hash(patient_id)) % (2**31)
    base_rng = np.random.RandomState(patient_base_seed)
    r_x = base_rng.randint(85, 105)
    r_y = base_rng.randint(95, 115)
    
    # Skull & Parenchyma
    cv2.ellipse(canvas, center, (r_x, r_y), 0, 0, 360, color=0.8, thickness=4)
    cv2.ellipse(canvas, center, (r_x - 5, r_y - 5), 0, 0, 360, color=0.45, thickness=-1)
    
    # Ventricles
    v_w = base_rng.randint(12, 22)
    v_h = base_rng.randint(25, 40)
    cv2.ellipse(canvas, (center[0] - 12, center[1]), (v_w // 2, v_h), -10, 0, 360, color=0.15, thickness=-1)
    cv2.ellipse(canvas, (center[0] + 12, center[1]), (v_w // 2, v_h), 10, 0, 360, color=0.15, thickness=-1)
    
    # Tumor pathology
    tumor_x = base_rng.randint(center[0] - 40, center[0] + 40)
    tumor_y = base_rng.randint(center[1] - 40, center[1] + 40)
    tumor_r = base_rng.randint(14, 24)
    
    if class_id == 0:
        cv2.circle(canvas, (tumor_x, tumor_y), tumor_r, color=0.95, thickness=-1)
    elif class_id == 1:
        cv2.circle(canvas, (tumor_x, tumor_y), tumor_r, color=0.75, thickness=-1)
        cv2.circle(canvas, (tumor_x, tumor_y), tumor_r // 2, color=0.25, thickness=-1)
    elif class_id == 2:
        cv2.ellipse(canvas, (center[0], center[1] + 50), (16, 12), 0, 0, 360, color=0.90, thickness=-1)

    noise = rng.normal(0, 0.03, canvas.shape).astype(np.float32)
    canvas = np.clip(canvas + noise, 0.0, 1.0)
    
    # Scanner black border padding
    pad_top = rng.randint(20, 45)
    pad_left = rng.randint(15, 35)
    framed = np.zeros((img_size + 64, img_size + 64), dtype=np.float32)
    framed[pad_top:pad_top + img_size, pad_left:pad_left + img_size] = canvas
    
    # Outlier spike (Rician noise simulation)
    framed[10, 10] = 12.0
    return framed


## 3. Patient-Level Data Leakage vs StratifiedGroupKFold
We construct the dataset manifest and compare standard random splitting against `StratifiedGroupKFold`.

In [ ]:
records = []
class_names = {0: "meningioma", 1: "glioma", 2: "pituitary"}

for p_num in range(1, 21):
    pid = f"PATIENT_{p_num:03d}"
    cls = p_num % 3
    for s_idx in range(1, 5):
        records.append({
            "slice_id": f"{pid}_slice_{s_idx}",
            "patient_id": pid,
            "label": cls,
            "label_name": class_names[cls],
            "slice_index": s_idx
        })

df = pd.DataFrame(records)
print(f"Total synthetic slices: {len(df)}, Unique patients: {df['patient_id'].nunique()}")

# 1. Random slice split
train_naive, test_naive = train_test_split(df, test_size=0.25, random_state=42, stratify=df['label'])
naive_train_patients = set(train_naive['patient_id'])
naive_test_patients = set(test_naive['patient_id'])
leaked = naive_test_patients.intersection(naive_train_patients)

print(f"[NAIVE RANDOM SPLIT] Test patients leaked into training set: {len(leaked)} / {len(naive_test_patients)} ({len(leaked)/len(naive_test_patients)*100:.1f}%)")

# 2. StratifiedGroupKFold split
sgkf = StratifiedGroupKFold(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df['label'], groups=df['patient_id'])):
    train_p = set(df.iloc[train_idx]['patient_id'])
    val_p = set(df.iloc[val_idx]['patient_id'])
    overlap = train_p.intersection(val_p)
    print(f"[STRATIFIED GROUP K-FOLD Fold {fold}] Train patients: {len(train_p)}, Val patients: {len(val_p)}, Overlap: {len(overlap)} (Zero Leakage)")


## 4. Cross-Dataset Contamination: MD5 vs 64-bit DCT pHash
We generate an original slice, simulate a JPEG compression + resize operation (typical of Kaggle collections), and compare MD5 vs pHash.

In [ ]:
orig_slice = generate_synthetic_slice("PATIENT_001", slice_idx=2, class_id=1)

# Simulate JPEG compression & 224x224 re-encoding
orig_u8 = (orig_slice * 255.0 / (orig_slice.max() + 1e-8)).astype(np.uint8)
_, enc = cv2.imencode('.jpg', orig_u8, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
reencoded_u8 = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE)
reencoded_slice = cv2.resize(reencoded_u8, (224, 224)).astype(np.float32) / 255.0

diff_slice = generate_synthetic_slice("PATIENT_009", slice_idx=1, class_id=0)
diff_u8 = (diff_slice * 255.0 / (diff_slice.max() + 1e-8)).astype(np.uint8)

# MD5 comparison
md5_orig = hashlib.md5(orig_u8.tobytes()).hexdigest()
md5_reencoded = hashlib.md5(reencoded_u8.tobytes()).hexdigest()
print("=== MD5 Cryptographic Hash ===")
print(f"Original MD5:   {md5_orig}")
print(f"Re-encoded MD5: {md5_reencoded}")
print(f"MD5 Collision:  {md5_orig == md5_reencoded} (Cryptographic hash misses re-encoded duplicate!)")

# pHash comparison
phash_orig = compute_phash(orig_slice)
phash_reencoded = compute_phash(reencoded_slice)
phash_diff = compute_phash(diff_slice)

dist_dup = hamming_distance(phash_orig, phash_reencoded)
dist_diff = hamming_distance(phash_orig, phash_diff)

print("\n=== Discrete Cosine Transform (pHash) ===")
print(f"Original pHash:          0x{phash_orig:016x}")
print(f"Re-encoded pHash:        0x{phash_reencoded:016x}")
print(f"Hamming Dist (Duplicate): {dist_dup} bits (<= 6 -> Detected as Duplicate)")
print(f"Hamming Dist (Different): {dist_diff} bits (> 6 -> Distinct Patient Scan)")


## 5. BK-Tree Sub-linear Duplicate Search
We index all synthetic slices into a BK-Tree and query it with the re-encoded image.

In [ ]:
tree = BKTree()
for idx, row in df.iterrows():
    s = generate_synthetic_slice(row['patient_id'], row['slice_index'], row['label'])
    h = compute_phash(s)
    tree.insert(h, row['slice_id'])

matches = tree.search(phash_reencoded, max_distance=6)
print(f"Indexed {tree.size} items in BK-Tree.")
print(f"BK-Tree query matches for re-encoded slice (Hamming <= 6):")
for item_id, dist in matches:
    print(f"  - Match: {item_id} (Hamming distance = {dist})")


## 6. Cranial Contour Isolation & Percentile Normalization
Visualizing Otsu contour bounding-box isolation and $[0.5, 99.5]\%$ percentile clipping.

In [ ]:
raw = generate_synthetic_slice("PATIENT_004", slice_idx=3, class_id=2)
cropped, bbox = crop_brain_contour(raw, margin_ratio=0.05)
norm = normalize_intensity(cropped)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(raw, cmap='bone')
axes[0].set_title(f"1. Raw Slice (With Borders & Spike)\nMax={raw.max():.1f}")
x, y, w, h = bbox
rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2)
axes[0].add_patch(rect)
axes[0].axis('off')

axes[1].imshow(cropped, cmap='bone')
axes[1].set_title(f"2. Otsu Cranial Crop\nBBox: ({x}, {y}, {w}, {h})")
axes[1].axis('off')

axes[2].imshow(norm, cmap='bone')
axes[2].set_title("3. Percentile Normalized [0.5, 99.5]%\nDynamic Range Standardized [0, 1]")
axes[2].axis('off')

plt.tight_layout()
plt.show()


## 7. Key Takeaways for Milestone 1
- **Patient Leakage**: Without grouping by `patient_id`, validation scores are dangerously inflated by anatomical and noise memorization.
- **Perceptual Deduplication**: `pHash` + `BKTree` finds lossy re-encoded copies across disparate clinical datasets.
- **Artifact Neutralization**: Otsu cropping removes scanner borders, and percentile normalization prevents Rician spikes from crushing image contrast.
